In [1]:
# ===== STANDALONE: harder eval (training ki zaroorat nahi) =====
!pip install -q gdown yacs
!pip install -q --no-deps git+https://github.com/KaiyangZhou/deep-person-reid.git

import os, json, glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

def find(name):
    hits = glob.glob(f"/kaggle/working/**/{name}", recursive=True) + \
           glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} nahi mila -> notebook output input mein attach karo"
    return hits[0]

SPLIT_P, CKPT_P = find("split.json"), find("reid_best.pt")
print("split:", SPLIT_P, "\nckpt :", CKPT_P)

split = json.load(open(SPLIT_P))

# agar dataset ka path badal gaya ho to image paths remap karo
if not os.path.exists(split["query"][0][0]):
    NEW_ROOT = next(p for p, d, _ in os.walk("/kaggle/input")
                    if any(x.lower().startswith("cam_") for x in d))
    OLD_ROOT = split["query"][0][0].split("/Cam_")[0]
    fix = lambda p: p.replace(OLD_ROOT, NEW_ROOT)
    split = {"train": {k: [fix(p) for p in v] for k, v in split["train"].items()},
             "query": [(fix(p), k) for p, k in split["query"]],
             "gallery": [(fix(p), k) for p, k in split["gallery"]]}
    print("paths remapped ->", NEW_ROOT)
assert os.path.exists(split["query"][0][0]), "image paths abhi bhi galat hain"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
H, W = 256, 128
eval_tf = transforms.Compose([transforms.Resize((H, W)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])

class EvalSet(Dataset):
    def __init__(s, pairs): s.p = pairs
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        path, key = s.p[i]
        return eval_tf(Image.open(path).convert("RGB")), key

class ReIDNet(nn.Module):
    def __init__(s, num_classes):
        super().__init__()
        import torchreid
        s.net = torchreid.models.build_model("osnet_x1_0", num_classes,
                                             loss="triplet", pretrained=True)
    def forward(s, x):
        out = s.net(x)
        return F.normalize(out[1] if s.training else out, dim=1)

ck = torch.load(CKPT_P, map_location=device, weights_only=False)
print("checkpoint epoch:", ck["epoch"], "| saved metrics:", ck["metrics"])

model_pre = ReIDNet(ck["num_classes"]).to(device)
model_tr  = ReIDNet(ck["num_classes"]).to(device)
model_tr.load_state_dict(ck["state_dict"])

# temporal-gap query/gallery
per_id = {}
for path, key in split["query"] + split["gallery"]:
    per_id.setdefault(key, []).append(path)

def frame_no(p):
    try: return int(os.path.splitext(os.path.basename(p))[0])
    except ValueError: return -1

q2, g2, skipped = [], [], 0
for key, paths in per_id.items():
    paths = sorted(paths, key=frame_no); n = len(paths)
    if n < 8: skipped += 1; continue
    k = max(1, n // 4)
    q2 += [(p, key) for p in paths[:k]]
    g2 += [(p, key) for p in paths[-k:]]
print(f"ids: {len(per_id)-skipped} | query: {len(q2)} | gallery: {len(g2)} | skipped: {skipped}")

q_dl = DataLoader(EvalSet(q2), batch_size=64, num_workers=2)
g_dl = DataLoader(EvalSet(g2), batch_size=64, num_workers=2)

@torch.no_grad()
def extract(m, dl):
    m.eval(); fs, ks = [], []
    for imgs, kk in dl:
        fs.append(m(imgs.to(device)).float().cpu()); ks += list(kk)
    return torch.cat(fs), np.array(ks)

def eval_pair(m):
    qf, qk = extract(m, q_dl); gf, gk = extract(m, g_dl)
    sim = (qf @ gf.t()).numpy()
    cmcs, aps = [], []
    for i in range(sim.shape[0]):
        o = np.argsort(-sim[i]); mt = (gk[o] == qk[i]).astype(np.int32)
        if mt.sum() == 0: continue
        cmcs.append(mt.cumsum().clip(max=1))
        pos = np.where(mt == 1)[0]
        aps.append(float(np.mean([(j+1)/(r+1) for j, r in enumerate(pos)])))
    cmc = np.mean(cmcs, axis=0)
    return {"Rank-1": float(cmc[0]), "Rank-5": float(cmc[4]), "mAP": float(np.mean(aps))}

r_pre, r_tr = eval_pair(model_pre), eval_pair(model_tr)
print("\ntemporal-gap eval (mushkil protocol):")
for k in ["Rank-1", "Rank-5", "mAP"]:
    print(f"  {k:7} pretrained {r_pre[k]*100:6.2f}%  ->  trained {r_tr[k]*100:6.2f}%"
          f"  ({(r_tr[k]-r_pre[k])*100:+.2f})")

os.makedirs("/kaggle/working/reid", exist_ok=True)
json.dump({"temporal_gap_eval": {"pretrained": r_pre, "trained": r_tr}},
          open("/kaggle/working/reid/reid_eval_hard.json", "w"), indent=2)
print("\nsaved: /kaggle/working/reid/reid_eval_hard.json")

  Preparing metadata (setup.py) ... done
split: /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/split.json 
ckpt : /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/reid_best.pt
checkpoint epoch: 20 | saved metrics: {'Rank-1': 1.0, 'Rank-5': 1.0, 'Rank-10': 1.0, 'mAP': 0.7457119876990418}


Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 181MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
ids: 55 | query: 1222 | gallery: 1222 | skipped: 3

temporal-gap eval (mushkil protocol):
  Rank-1  pretrained  29.71%  ->  trained  61.62%  (+31.91)
  Rank-5  pretrained  39.93%  ->  trained  75.61%  (+35.68)
  mAP     pretrained  23.60%  ->  trained  55.71%  (+32.12)

saved: /kaggle/working/reid/reid_eval_hard.json


In [3]:
import glob
print(glob.glob("/kaggle/working/reid/*"))

['/kaggle/working/reid/reid_eval_hard.json']
